# 下見：GSM-Symbolic P1・P2 の難しさ（Haiku 4.5・Claude API・CPU）

**探索の下見です（ラベルなし）。** 決まりは `notes/hard_prelook_plan.md`（リクエストを送る前に書いたもの）。

- P1・P2 のテンプレートごとに 2 例。1 例につき、土台の道 1 回（温度 0）と引き直し 4 回（プリフィルなし）を、CR-1a と同じ頼み方（60 語）で解かせます。
- 出すもの：正答率、引き直しで答えが変わる割合、テンプレートごとの正答率の分布、長さ、**本番をやったときの費用の見込み**。
- 問題文・応答はどこにも表示・保存しません（数とハッシュだけ）。
- 費用は **$1.5 前後**の見込みで、**$3** に近づいたら止まります。途中で切れても、もう一度「すべてのセルを実行」で続きから回ります（保存ずみの例は呼び直しません）。

**使い方**：CPU のランタイム。シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。「すべてのセルを実行」で 10〜20 分くらい。結果は Drive の `DECISION_POINTS/prelook/HARD_PRELOOK_summary.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned SDK, key (never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
EXPECTED_HEAD = "f92d0fb20062b0e644a725c8fd722bbf8098f9d3"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
assert (REPO / "notes/hard_prelook_plan.md").exists()
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import hard_prelook as H
from pork import cr1_requests as RQ
from pork import cr1_analysis as A
from pork import cr1_run as R
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RQ.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RQ.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key, max_retries=0)
del _key
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/prelook")
OUT.mkdir(parents=True, exist_ok=True)
print("exact head:", head, "| prompt:", H.STUDY_PROMPT, RQ.prompt_sha256(H.STUDY_PROMPT)[:12], "| model:", A.MODEL, "| anthropic", anthropic.__version__)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = '下見 GSM-Symbolic P1・P2 の難しさ（Haiku 4.5・API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - Hard prelook P1 P2', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — pick, call (resumable), summarize.  Numbers and hashes only.
t0 = time.time()
jobs, data_ids = [], {}
for config in H.CONFIGS:
    rows, info = H.load_config_rows(config)
    data_ids[config] = info
    jobs += H.pick(rows, config)
    del rows
print("例の数:", {c: sum(j["config"] == c for j in jobs) for c in H.CONFIGS}, "| 呼び出しの予定:", len(jobs) * (1 + H.N_SAMPLES))
HDR = {"code_head": head, "prompt_sha256": RQ.prompt_sha256(H.STUDY_PROMPT), "model": A.MODEL, "sdk": anthropic.__version__,
       "datasets": data_ids, "seed": H.SEED_PRELOOK, "n_inst": H.N_INST, "n_samples": H.N_SAMPLES,
       "jobs_sha256": hashlib.sha256(json.dumps([H.job_key(j) for j in jobs]).encode()).hexdigest()}
STATE = OUT / "HARD_PRELOOK_state.json"
if STATE.exists():
    st = json.loads(STATE.read_text())
    assert st["header"] == HDR, "別の実行の記録です。止めます（Claude に教えてね）。"
    print("続きから：", len(st["done"]), "例ずみ | $%.3f" % st["spent_usd"])
else:
    st = H.new_state(HDR)
_lock_save = __import__("threading").Lock()
def save():
    with _lock_save:
        R._atomic_write(STATE, st)
RETRY_ON = (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError)
n = H.run(jobs, lambda p: client.messages.create(**p), st, save, retry_on=RETRY_ON)
save()
summary = H.summarize(st)
summary["header"] = HDR
summary["seconds"] = round(time.time() - t0, 1)
R._atomic_write(OUT / "HARD_PRELOOK_summary.json", summary)
for c in H.CONFIGS:
    s = summary.get(c)
    if s:
        print(c, "| テンプレート", s["templates"], "| 土台の正答率 %.3f" % s["baseline_accuracy"], "| q0 %.3f" % s["q0_mean"],
              "| 0.2〜0.8 のテンプレート", s["templates_in_band"], "| 100 問の本番の見込み $%s" % (None if s["estimate_main_run"]["usd_100_items"] is None else round(s["estimate_main_run"]["usd_100_items"], 1)))
print("決め方:", summary["decision"]["route"], "| 今回の呼び出し:", n, "| 費用 $%.3f" % st["spent_usd"])
notify('終わりました：下見（P1・P2 の難しさ）の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()